# MCP — a weather MCP server, a client, and an agent that uses it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/SA_LIB_WORKSHOP_Day2/blob/main/09_mcp/09_mcp.ipynb)

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

**MCP (Model Context Protocol)** is a standard plug for AI tools. `weather_server.py` wraps a free public API (Open-Meteo, no key) as MCP tools; `01` calls them directly (no LLM, no key); `02` is **one agent** that discovers the tools at run time and lets the LLM decide when to use them.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "mcp>=2.0" "fastmcp>=4.0" "httpx" "openai" "python-dotenv" "ddgs"

### API keys
* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

### helper file: `weather_server.py`
a simple MCP SERVER that wraps a real public API (Open-Meteo weather — free, no key)

MCP (Model Context Protocol) is a standard plug for AI tools: write your tools ONCE as an MCP
server, and any MCP client can use them — our agent, Claude Desktop, VS Code, Cursor, the
OpenAI Agents SDK, Google ADK, LangGraph ...

this file is started BY the client (01_mcp_client.py / 02_mcp_agent.py). it talks over
stdin/stdout ("stdio" transport), so you normally don't run it by hand.
to test it on its own:  python 09_mcp/weather_server.py   (it waits silently for a client; Ctrl+C to stop)

In [ ]:
%%writefile weather_server.py
# a simple MCP SERVER that wraps a real public API (Open-Meteo weather — free, no key)
#
# MCP (Model Context Protocol) is a standard plug for AI tools: write your tools ONCE as an MCP
# server, and any MCP client can use them — our agent, Claude Desktop, VS Code, Cursor, the
# OpenAI Agents SDK, Google ADK, LangGraph ...
#
# this file is started BY the client (01_mcp_client.py / 02_mcp_agent.py). it talks over
# stdin/stdout ("stdio" transport), so you normally don't run it by hand.
# to test it on its own:  python 09_mcp/weather_server.py   (it waits silently for a client; Ctrl+C to stop)

import logging

import httpx
from mcp.server.mcpserver import MCPServer

logging.getLogger("httpx").setLevel(logging.WARNING)   # keep the output clean

server = MCPServer("weather")   # the name clients will see

GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
WEATHER_CODES = {0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast", 45: "fog", 51: "light drizzle",
                 61: "light rain", 63: "rain", 65: "heavy rain", 80: "rain showers", 95: "thunderstorm"}


def find_city(city: str) -> dict | None:
    """Step 1 of the API: city name -> latitude / longitude."""
    results = httpx.get(GEOCODE_URL, params={"name": city, "count": 1}, timeout=15).json().get("results")
    return results[0] if results else None


# every @server.tool() becomes a tool that MCP clients can discover and call.
# the docstring and type hints are what the client (and the LLM) sees.
@server.tool()
def get_current_weather(city: str) -> str:
    """Get the current weather (temperature, wind, conditions) for a city, e.g. 'Tripoli' or 'Benghazi'."""
    place = find_city(city)
    if not place:
        return f"I could not find a city called '{city}'."
    now = httpx.get(WEATHER_URL, timeout=15, params={
        "latitude": place["latitude"], "longitude": place["longitude"],
        "current": "temperature_2m,wind_speed_10m,weather_code"}).json()["current"]
    sky = WEATHER_CODES.get(now["weather_code"], f"weather code {now['weather_code']}")
    return (f"{place['name']}, {place.get('country', '')}: {now['temperature_2m']} °C, "
            f"wind {now['wind_speed_10m']} km/h, {sky} (at {now['time']} UTC)")


@server.tool()
def get_forecast(city: str, days: int = 3) -> str:
    """Get the daily min/max temperature forecast for a city for the next 1-7 days."""
    place = find_city(city)
    if not place:
        return f"I could not find a city called '{city}'."
    daily = httpx.get(WEATHER_URL, timeout=15, params={
        "latitude": place["latitude"], "longitude": place["longitude"], "timezone": "auto",
        "daily": "temperature_2m_min,temperature_2m_max", "forecast_days": max(1, min(days, 7))}).json()["daily"]
    rows = [f"{d}: {lo}–{hi} °C" for d, lo, hi in zip(daily["time"], daily["temperature_2m_min"], daily["temperature_2m_max"])]
    return f"{place['name']} forecast:\n" + "\n".join(rows)


@server.tool()
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


if __name__ == "__main__":
    server.run("stdio")   # other transports: "streamable-http" (a web server other machines can reach)

## 1. `01_mcp_client.py`

MCP 01 — MCP + API, no LLM: connect to the MCP server, list its tools, call them
this is what every MCP client does under the hood (Claude Desktop, VS Code, agent frameworks):
  1. START / CONNECT to the server       2. INITIALIZE (handshake)
  3. LIST TOOLS (names, descriptions, input schemas)       4. CALL TOOLS

In [ ]:
import asyncio
import json
import sys
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command=sys.executable, args=[str(Path.cwd() / "weather_server.py")])


async def main():
    async with stdio_client(SERVER) as (read, write):          # 1. start the server as a subprocess
        async with ClientSession(read, write) as session:
            await session.initialize()                          # 2. handshake

            tools = (await session.list_tools()).tools         # 3. what can this server do?
            print("tools offered by the server:")
            for t in tools:
                print(f"  - {t.name}: {t.description}")
                print(f"      input schema: {json.dumps(t.input_schema.get('properties', {}))}")

            for name, args in [("get_current_weather", {"city": "Tripoli"}),     # 4. call them
                               ("get_forecast", {"city": "Benghazi", "days": 3}),
                               ("word_count", {"text": "MCP is a standard plug for AI tools"})]:
                result = await session.call_tool(name, args)
                print(f"\n>>> {name}({args})\n{result.content[0].text}")


if __name__ == "__main__":
    await main()

## 2. `02_mcp_agent.py`

MCP 02 — MCP + API + ONE AGENT
the agent does NOT define any tools itself. it asks the MCP server what tools exist, gives them
to the LLM, and when the LLM wants a tool, the agent forwards the call to the MCP server.

  user --> [agent + LLM] --tool call--> [MCP client] --stdio--> [weather_server.py] --HTTP--> Open-Meteo API

swap weather_server.py for ANY other MCP server (files, GitHub, databases, ...) and the agent
gets new abilities without changing a line of agent code. that is the point of MCP.

In [ ]:
import asyncio
import json
import sys
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import OpenAI

from workshop_common import get_llm_config

SERVER = StdioServerParameters(command=sys.executable, args=[str(Path.cwd() / "weather_server.py")])
cfg = get_llm_config()
llm = OpenAI(base_url=cfg.base_url, api_key=cfg.api_key)


def mcp_to_openai(tool) -> dict:
    """An MCP tool description -> the 'function calling' format every LLM provider understands."""
    return {"type": "function", "function": {
        "name": tool.name, "description": tool.description or "", "parameters": tool.input_schema}}


async def run_agent(session: ClientSession, question: str, max_steps: int = 6) -> str:
    tools = [mcp_to_openai(t) for t in (await session.list_tools()).tools]    # discover tools at run time
    messages = [{"role": "system", "content": "You are a helpful weather assistant. Use your tools for facts. Be concise."},
                {"role": "user", "content": question}]
    for _ in range(max_steps):                                                 # plan -> act -> observe
        msg = llm.chat.completions.create(model=cfg.model, messages=messages, tools=tools).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            print(f"  [agent -> MCP] {call.function.name}({args})")
            result = await session.call_tool(call.function.name, args)          # the MCP server does the work
            text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": text})
    return "stopped: too many steps"


async def main():
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            for question in ["What's the weather like in Tripoli right now?",
                             "Compare the next 3 days in Benghazi and Sabha. Which city will be hotter?"]:
                print(f"\nuser: {question}")
                print(f"agent: {await run_agent(session, question)}")


if __name__ == "__main__":
    await main()

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*